In [ ]:
%run ../shared_notebook_setup.py

## System Prompts & Personas

The `system` role message sets persistent instructions for the model — useful for defining tone, role, language, or constraints.

In [3]:
response = client.chat.completions.create(
    model=DATABRICKS_ENDPOINT,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a pirate assistant. You answer every question helpfully "
                "but always respond in pirate-speak, using 'Arrr', 'matey', 'ye', etc."
            ),
        },
        {
            "role": "user",
            "content": "What is the capital of France?",
        },
    ],
)

print(response.choices[0].message.content)

Ye be askin' a simple question, matey! Arrr, the capital o' France be Paris, the City o' Light! Aye, it's a grand place, full o' treasure and booty... I mean, art and history! Ye should set sail fer Paris, me hearty, and explore the Eiffel Tower, the Louvre, and all the other wonders it has to offer! Arrr!


## Multi-Turn Conversation

Maintain conversation context by appending each assistant reply back into the `messages` list before the next user turn.

In [4]:
history = [
    {"role": "system", "content": "You are a helpful assistant that remembers the conversation."},
]

def chat(user_input: str) -> str:
    history.append({"role": "user", "content": user_input})
    resp = client.chat.completions.create(
        model=DATABRICKS_ENDPOINT,
        messages=history,
    )
    reply = resp.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply

# Turn 1
print("User: What is photosynthesis?")
print("Assistant:", chat("What is photosynthesis?"))
print()

# Turn 2 — model should remember what we were just talking about
print("User: Can you give me a one-sentence summary of what you just said?")
print("Assistant:", chat("Can you give me a one-sentence summary of what you just said?"))

User: What is photosynthesis?
Assistant: Photosynthesis is the process by which plants, algae, and some bacteria convert light energy from the sun into chemical energy in the form of glucose (a type of sugar). This process is essential for life on Earth, as it provides energy and organic compounds for plants to grow and thrive, and ultimately supports the food chain.

Here's a simplified overview of the photosynthesis process:

**Inputs:**

1. Light energy from the sun
2. Carbon dioxide (CO2) from the atmosphere
3. Water (H2O) from the soil or surrounding environment

**Outputs:**

1. Glucose (C6H12O6), a type of sugar that serves as energy storage for the plant
2. Oxygen (O2), released into the atmosphere as a byproduct

**The Process:**

1. Light is absorbed by pigments such as chlorophyll and other accessory pigments in the plant's leaves.
2. The energy from light is used to power a series of chemical reactions that convert CO2 and H2O into glucose and O2.
3. The overall equation fo

## Streaming

With `stream=True` the API returns an iterator of chunks. This lets you display tokens in real-time instead of waiting for the full response.

In [5]:
import sys

stream = client.chat.completions.create(
    model=DATABRICKS_ENDPOINT,
    messages=[
        {"role": "user", "content": "Write a short poem about machine learning."},
    ],
    stream=True,
)

print("Streaming response (tokens arrive one by one):\n")
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
print()  # newline after stream ends

Streaming response (tokens arrive one by one):

Here is a short poem about machine learning:

Circuits wired, data fed,
A mind is born, from code ahead.
It learns, adapts, and grows with ease,
A digital brain, with neural peace.

With each example, it refines its art,
Predicting outcomes, a calculated start.
Through trial and error, it finds its way,
To make decisions, come what may.

In silicon halls, a new mind stirs,
A machine that learns, and innovates and blurs,
The lines between man and machine so fine,
A future unfolds, with AI's design.


## Temperature & Parameter Control

| Parameter | Effect |
|-----------|--------|
| `temperature` | 0 = deterministic, 1+ = more creative/random |
| `max_tokens` | Hard cap on output length |
| `top_p` | Nucleus sampling; alternative to temperature |
| `n` | Number of independent completions to generate |

In [6]:
prompt = "Give me a nice product slogan for a coffee brand."

for temp in [0.0, 0.7, 1.4]:
    resp = client.chat.completions.create(
        model=DATABRICKS_ENDPOINT,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        max_tokens=60,
    )
    print(f"temperature={temp}: {resp.choices[0].message.content.strip()}")

temperature=0.0: Here are a few ideas for a coffee brand slogan:

1. **"Brewing a better day, one cup at a time."** - a friendly, uplifting message that positions your coffee as a daily pick-me-up.
2. **"Fuel your passion, one sip at a time."
temperature=0.7: A great cup of coffee deserves a great slogan! Here are a few ideas for a coffee brand:

1. **"Brewing moments that matter"** - a heartwarming slogan that evokes the idea of coffee being a part of special moments.
2. **"Fuel for the everyday hero
temperature=1.4: Here are a few ideas:

1. **"Fuel Your Day, Naturally."** - simple, straightforward, and conveying a sense of wholesomeness.
2. **"Brew Love, One Cup at a Time."** - evoking a sense of warmth and joy.
3. **"


## 7 — Structured / JSON Output

Ask the model to return a strict JSON object. Setting `response_format={"type": "json_object"}` (where supported) or simply instructing via the system prompt forces machine-readable output.

In [7]:
import json

resp = client.chat.completions.create(
    model=DATABRICKS_ENDPOINT,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a data extraction assistant. "
                "Always respond with valid JSON only — no markdown, no prose. "
                "Extract the requested fields from the user's text."
            ),
        },
        {
            "role": "user",
            "content": (
                "Extract the person's name, age, and city from this sentence: "
                "'Sarah, who is 29 years old, recently moved to Austin.'"
                "\n\nRespond as JSON with keys: name, age, city."
            ),
        },
    ],
    temperature=0,
)

raw = resp.choices[0].message.content
parsed = json.loads(raw)

print("Raw response :", raw)
print("Parsed Python :", parsed)
print("Name:", parsed["name"], "| Age:", parsed["age"], "| City:", parsed["city"])

Raw response : {"name": "Sarah", "age": 29, "city": "Austin"}
Parsed Python : {'name': 'Sarah', 'age': 29, 'city': 'Austin'}
Name: Sarah | Age: 29 | City: Austin
